**Data Info**

train.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용
* first_party_winner : 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

test.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용

sample_submission.csv - 제출 양식
* ID : 사건 샘플 ID
* first_party_winner : 예측한 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

# Baseline: TF-IDF + LightGBM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import hstack
import lightgbm as lgb

In [ ]:
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

In [ ]:
# Data preprocessing
vec_facts = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
vec_party = TfidfVectorizer(max_features=2000)  # 이름은 어휘가 작으니 차원도 작게

def get_vector(df, train_mode):
    if train_mode:
        X_facts = vec_facts.fit_transform(df['facts'])
        X_p1 = vec_party.fit_transform(pd.concat([df['first_party'], df['second_party']]))
        X_p1 = vec_party.transform(df['first_party'])
        X_p2 = vec_party.transform(df['second_party'])
    else:
        X_facts = vec_facts.transform(df['facts'])
        X_p1 = vec_party.transform(df['first_party'])
        X_p2 = vec_party.transform(df['second_party'])
    return hstack([X_p1, X_p2, X_facts]).tocsr()  # sparse 유지 (dense 변환 X)

X = get_vector(train, True)
y = train["first_party_winner"].values
X_test = get_vector(test, False)

In [ ]:
# Basemodel: 5-foldCV + LightGBM
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_pred = np.zeros(len(train))
test_pred = np.zeros(len(test))

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "is_unbalance": True,
    "verbosity": -1,
    "seed": 42,
}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    dtrain = lgb.Dataset(X[tr_idx], label=y[tr_idx])
    dvalid = lgb.Dataset(X[va_idx], label=y[va_idx], reference=dtrain)

    model = lgb.train(
        params, dtrain, num_boost_round=2000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    va_pred = model.predict(X[va_idx], num_iteration=model.best_iteration)
    oof_pred[va_idx] = va_pred
    test_pred += model.predict(X_test, num_iteration=model.best_iteration) / skf.n_splits

    acc = accuracy_score(y[va_idx], (va_pred > 0.5).astype(int))
    print(f"Fold {fold+1} | Acc: {acc:.4f} | best_iter: {model.best_iteration}")

print("\nOverall OOF Accuracy:", accuracy_score(y, (oof_pred > 0.5).astype(int)))
print("Overall OOF Macro F1:", f1_score(y, (oof_pred > 0.5).astype(int), average="macro"))


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[8]	valid_0's binary_logloss: 0.630871
Fold 1 | Acc: 0.6653 | best_iter: 8
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2]	valid_0's binary_logloss: 0.633948
Fold 2 | Acc: 0.6653 | best_iter: 2
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[14]	valid_0's binary_logloss: 0.625445
Fold 3 | Acc: 0.6653 | best_iter: 14
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[12]	valid_0's binary_logloss: 0.628617
Fold 4 | Acc: 0.6667 | best_iter: 12
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[5]	valid_0's binary_logloss: 0.632133
Fold 5 | Acc: 0.6646 | best_iter: 5

Overall OOF Accuracy: 0.66545601291364
Overall OOF Macro F1: 0.3995638478313545


In [ ]:
# Submission
submit = pd.read_csv('./sample_submission.csv')
submit['first_party_winner'] = (test_pred > 0.5).astype(int)
submit.to_csv('./baseline-lightGBM_submit.csv', index=False)
print('Done')

Done
